In [ ]:
import duckdb

# DuckDB is incredibly fast and memory efficient
duckdb.sql("""
    COPY (
        SELECT * FROM read_parquet('your_file.parquet')
    ) TO 'half_1.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 100000);
""")

# For second half, you'd need to do it differently
# But this shows the speed potential

In [1]:
import duckdb
import os

# Your file path
file_path = '../../../data/raw/tables/original/lineitem.parquet'

# First, get total row count (fast, doesn't load data)
total_rows = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{file_path}')").fetchone()[0]
half_rows = total_rows // 2

print(f"Total rows: {total_rows:,}")
print(f"Split point: {half_rows:,} rows")

# Split into two halves (fast and memory efficient)
print("Creating first half...")
duckdb.sql(f"""
    COPY (
        SELECT * FROM read_parquet('{file_path}')
        LIMIT {half_rows}
    ) TO 'half_1.parquet' (FORMAT PARQUET, COMPRESSION 'SNAPPY');
""")

print("Creating second half...")
duckdb.sql(f"""
    COPY (
        SELECT * FROM read_parquet('{file_path}')
        OFFSET {half_rows}
    ) TO 'half_2.parquet' (FORMAT PARQUET, COMPRESSION 'SNAPPY');
""")

# Verify the splits
size1 = os.path.getsize('half_1.parquet') / 1024 / 1024
size2 = os.path.getsize('half_2.parquet') / 1024 / 1024
rows1 = duckdb.sql("SELECT COUNT(*) FROM read_parquet('half_1.parquet')").fetchone()[0]
rows2 = duckdb.sql("SELECT COUNT(*) FROM read_parquet('half_2.parquet')").fetchone()[0]

print(f"\n✓ Split complete!")
print(f"  half_1.parquet: {rows1:,} rows ({size1:.1f} MB)")
print(f"  half_2.parquet: {rows2:,} rows ({size2:.1f} MB)")
print(f"  Total: {rows1 + rows2:,} rows (matches original: {total_rows:,})")

Total rows: 6,001,215
Split point: 3,000,607 rows
Creating first half...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Creating second half...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✓ Split complete!
  half_1.parquet: 3,000,607 rows (131.1 MB)
  half_2.parquet: 3,000,608 rows (130.9 MB)
  Total: 6,001,215 rows (matches original: 6,001,215)


In [2]:
import os
print(f"Current working directory: {os.getcwd()}")

# The files will be created here if you don't specify a path

Current working directory: /home/jovyan/work/3) Final Dance/4) Local Bronze Layer to SnowFlake
